In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.bm25 import BM25Retriever

meta_df = pd.read_csv('../data/processed/meta_sample_processed.csv')
reviews_df = pd.read_csv('../data/processed/reviews_sample_processed.csv')

merged_df = pd.merge(
    reviews_df, 
    meta_df[['parent_asin', 'title']], 
    on='parent_asin', 
    how='left', 
    suffixes=('_review', '_product')
)

merged_df['title_product'] = merged_df['title_product'].fillna("Title Unavailable in Sample")

In [2]:
corpus_records = []
for _, row in merged_df.iterrows():
    record = {
        'parent_asin': row['parent_asin'],
        'retrieval_text': str(row['retrieval_text']), # Ensure it's a string
        'product_title': str(row['title_product']),   
        'review_text': str(row['text_review'] if 'text_review' in row else row['text']),              
        'rating': row['rating']                  
    }
    corpus_records.append(record)

In [3]:
retriever = BM25Retriever(
    index_path='../data/processed/bm25_index.pkl',
    corpus_path='../data/processed/corpus_data.pkl'
)

retriever.build_and_save_index(corpus_records)

Tokenizing corpus...
Building BM25 Index...
Index successfully saved to ../data/processed/bm25_index.pkl


In [4]:
retriever.load_index()
results = retriever.search("great software", top_n=1)

print("\n--- TEST SEARCH RESULTS ---")
for doc, score in results:
    print(f"Score: {score:.2f}")
    print(f"Product: {doc['product_title']}")
    print(f"Rating: {doc['rating']} Stars")
    print(f"Review: {doc['review_text'][:100]}...")


--- TEST SEARCH RESULTS ---
Score: 1.38
Product: Days Of Thunder [Blu-ray]
Rating: 5.0 Stars
Review: Gift for son.  Great price for blue ray....
